# Ingesta de datos históricos de sequía en Sonora

Esta libreta de Jupyter se utilizará para la ingesta y limpieza de los datos históricos de sequía en Sonora. A partir de los conjuntos de datos públicos de la CONAGUA, se utilizaran las variables de intensidad, frecuencia y temporalidad a nivel municipal a lo largo de los años. El pipeline contempla el proceso y transformación de los datos para facilitar su análisis exploratorio y visualización.

## Ingesta de datos

### Librerías necesarias

In [1]:
#!pip install -q pandas requests pydantic

In [2]:
import pandas as pd
from pydantic import ValidationError

# Definir rutas y parámetros
from sonora_health_climate_eda.config import (
    INTERIM_DATA_DIR,
    PANEL_END_YEAR,
    PANEL_START_YEAR,
)
from sonora_health_climate_eda.dataset import download_drought_data # importar la función de descarga de datos de sequía
from sonora_health_climate_eda.schemas import SequiaSonoraSchema # improtar el esquema de validación de datos de sequía

# Carpeta donde se guardan los CSV de sequía (data/interim/sonora_sequia en la raíz)
output_dir = INTERIM_DATA_DIR / "sonora_sequia"

## Descarga de datos

In [3]:
download_drought_data(PANEL_START_YEAR, PANEL_END_YEAR)

2026-11-12 20:07:56.033 | INFO     | sonora_health_climate_eda.dataset:download_drought_data:47 - Iniciando descarga de datos de sequía desde: https://smn.conagua.gob.mx/tools/RESOURCES/Monitor de Sequia en Mexico/MunicipiosSequia.xlsx
2026-11-12 20:08:05.352 | SUCCESS  | sonora_health_climate_eda.dataset:download_drought_data:51 - ¡Archivo descargado y guardado con éxito en: C:\Users\victo\OneDrive\Documentos Personales\GitHub\sonora-agriclimate-eda\data\raw\MunicipiosSequia.xlsx!
2026-11-12 20:08:05.353 | INFO     | sonora_health_climate_eda.dataset:download_drought_data:60 - Leyendo el archivo crudo de CONAGUA desde C:\Users\victo\OneDrive\Documentos Personales\GitHub\sonora-agriclimate-eda\data\raw\MunicipiosSequia.xlsx...
2026-11-12 20:08:11.806 | INFO     | sonora_health_climate_eda.dataset:download_drought_data:87 - Transformando a formato largo (Tidy Data)...
2026-11-12 20:08:12.003 | SUCCESS  | sonora_health_climate_eda.dataset:download_drought_data:109 - Año 2015: Descargado 

## Validación de la descarga de datos

### Función para validar los datos descargados

In [4]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            agriculture = SequiaSonoraSchema(**row.to_dict())
            validated_rows.append(agriculture.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [5]:
# Leer TODOS los años descargados y unirlos en un solo DataFrame
archivos = sorted(output_dir.glob("sonora_sequia_*.csv"))
df_sequia = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)
print(f"{len(archivos)} archivos, {len(df_sequia):,} filas")

6 archivos, 10,368 filas


In [6]:
df_validated = validate_df(df_sequia)
print(f"Filas válidas: {len(df_validated):,} de {len(df_sequia):,}")
df_validated.head(5)

Filas válidas: 10,368 de 10,368


,CVE_CONCATENADA,CVE_ENT,CVE_MUN,fecha,categoria_sequia,Anio,severidad_num
0,26001,26,1,2015-01-15,NaN,2015,0
1,26002,26,2,2015-01-15,D0,2015,1
2,26003,26,3,2015-01-15,D0,2015,1
3,26004,26,4,2015-01-15,D0,2015,1
4,26005,26,5,2015-01-15,D0,2015,1
